# 📗 그래프 알고리즘: 커뮤니티 탐지

33일차에서는 그래프를 **투영**하고 **PageRank** 로 "누가 중요한가"를 점수로 매겼습니다. 이번 시간에는 시선을 하나에서 **무리**로 옮깁니다. 서로 촘촘히 뭉친 무리를 찾아내는 **커뮤니티 탐지**입니다.

오늘 쓰는 데이터는 **아시아 항공 노선망**입니다. 공항 767곳과 그 사이 노선을 담았습니다. 여기에는 다른 실습에 없는 것이 하나 있습니다. **정답이 붙어 있습니다.** 공항마다 어느 국가·지역에 있는지가 데이터에 들어 있어서(47개), 알고리즘이 찾아낸 무리가 **실제 지도와 얼마나 맞는지 숫자로 확인**할 수 있습니다. 현장에서 커뮤니티 탐지를 쓸 때 반드시 하는 검증이고, 오늘 실습의 중심입니다.

## ⏪ 복습: 여기까지 배운 것

- **GDS 투영**: 33일차에서 분석 전용 엔진 GDS 를 쓰려면 그래프를 메모리에 **투영**(분석용 사본)해야 한다고 배웠습니다. `gds.graph.project` 로 만들고 `gds.graph.list`·`gds.graph.drop` 으로 관리했습니다.
- **PageRank·매개 중심성**: 33일차에서 노드 하나의 중요도를 점수로 매기고, 점수는 상대값이라 **순위**로 읽었습니다.
- **인덱스·제약**: 31일차에서 배운 대로, 많은 행을 적재할 때 유일성 제약이 없으면 한 줄마다 전체를 훑습니다. 오늘 적재 셀도 제약을 먼저 겁니다.
- 오늘은 같은 투영 위에서 **커뮤니티 탐지** 알고리즘을 돌립니다.

**오늘의 목표**

**1. 커뮤니티와 무방향 투영**
- [ ] (1-1) 노선망을 **UNDIRECTED(무방향)** 로 투영하는 이유를 알고, **차수와 상대 공항 수**가 다른 값이라는 것을 안다.

**2. Leiden 으로 커뮤니티 찾기**
- [ ] (2-1) **Leiden** 으로 묶음을 찾고(`stream`), 크기 분포로 결과를 먼저 훑는다.
- [ ] (2-2) 찾은 묶음을 `write` 로 그래프에 저장한다.

**3. 정답과 대조하기**
- [ ] (3-1) 찾은 묶음을 **실제 국가·지역과 대조**해 **교차표**로 눈으로 본다.
- [ ] (3-2) **순도**를 읽는다.
- [ ] (3-3) **NMI** 로 전체를 한 숫자로 요약한다.

**4. 알고리즘 견주기**
- [ ] (4-1) **Louvain·라벨 전파**와 나란히 놓고 **모듈러리티**로 어느 분할이 나은지 판단한다.
- [ ] (4-2) 관계에 붙은 숫자를 `relationshipWeightProperty` 로 넘기는 법과, 커뮤니티 탐지와 최단 경로에서 그 값의 **뜻이 반대**라는 것을 안다.

**5. 흔들림과 해상도**
- [ ] (5-1) 결과가 **실행마다 흔들린다**는 것을 확인한다.
- [ ] (5-2) 재현하는 방법과 그 **한계**를 다룬다.
- [ ] (5-3) **해상도(gamma)** 를 다룬다.

**6. 큰 그래프를 그림으로**
- [ ] (6-1) 공항 767곳짜리 그래프를 **읽을 수 있는 그림**으로 줄인다(크기 막대·지도 산점도).
- [ ] (6-2) **메타그래프**로 묶음 사이의 흐름을 본다.

아래 준비 셀 여섯 개를 **위에서부터** 실행하세요. 연결 → 그래프 초기화 → 투영 정리 → 항공 노선망 적재 → 무방향 투영 → 따라하기용 메일망 적재 순서입니다. Neo4j 는 반드시 **실습 전용 DB**에 연결하세요(아래 실습이 그래프를 지우고 새로 만듭니다).

다섯 번째 셀이 만드는 **투영**은 1-1 에서 한 번 더 만듭니다. 거기서는 **어떻게 만드는지**를 보는 것이 목적이라, 같은 투영을 내리고 다시 만듭니다.

In [ ]:
# [제공 코드] Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 반드시 "실습 전용" 데이터베이스에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# os.getenv(키, 기본값): .env 를 못 읽어도 에러가 아니라 이 기본값으로 조용히 넘어간다.
# 그러니 이 셀 마지막 줄에 찍히는 주소가 "실습 전용 DB" 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

In [ ]:
# [제공 코드] 실습 전용 DB 초기화: 지우기 전에 이 DB 가 맞는지 먼저 확인합니다.
UNIT_LABELS = ['Airport', 'Compound', 'DayType', 'Disease', 'ExDenseEvent', 'ExDisease', 'ExDomestic', 'ExDrug', 'ExEvent', 'ExForeign', 'ExIdKey', 'ExKing', 'ExKinng', 'ExNameKey', 'ExNodeKeyDemo', 'ExPerson', 'ExReign', 'ExScopeEvent', 'ExThrone', 'ExUniqueDemo', 'ExWorld', 'ExYear', 'FlatDrug', 'FlatKing', 'FlatOrder', 'Gene', 'IdKey', 'Line', 'Member', 'NameKey', 'NodeClass', 'NodeDay', 'NodeDrug', 'NodeKing', 'NodeMonth', 'NodeOrder', 'NodeYear', 'Person', 'PharmacologicClass', 'Station', 'SurveyDay', 'SurveyYear', 'Symptom', 'TempStation', 'TryDay', 'TryLine', 'TryStation']   # 이 단원이 만드는 레이블 전부(앞 일차가 남긴 것까지)

# 이 단원 것이 아닌 노드가 하나라도 있으면 지우지 않고 멈춥니다.
# .env 의 주소가 어긋나도 접속은 조용히 성공하므로, 지우기 전에 확인하는 수밖에 없습니다.
# all 로 보는 이유: any 로 보면 :Person:PatientRecord 처럼 한 레이블만 겹치는 남의 노드가 통과합니다
foreign = run_cypher("""
MATCH (n) WHERE size(labels(n)) = 0
   OR NOT all(label IN labels(n) WHERE label IN $unit_labels)
RETURN DISTINCT labels(n) AS labels LIMIT 5""", unit_labels=UNIT_LABELS)
if foreign:
    raise RuntimeError(
        f"{NEO4J_URI} 에 이 단원 것이 아닌 노드가 있습니다: {foreign}\n"
        "다른 실습이나 개인 데이터가 든 DB 로 보여 초기화를 멈췄습니다.\n"
        ".env 의 NEO4J_URI 가 실습 전용 DB 를 가리키는지 먼저 확인하세요.\n"
        "주소가 맞다면 위 레이블은 앞 실습이 남긴 것입니다. UNIT_LABELS 에 더하고 다시 실행하세요.")

# 여기까지 왔으면 이 DB 에는 이 단원이 만든 노드밖에 없습니다.
run_cypher("MATCH (n) DETACH DELETE n")   # DETACH: 노드에 붙은 관계까지 함께 지운다

print("초기화 완료:", NEO4J_URI, "· 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"])

In [ ]:
# [제공 코드] 남아 있는 GDS 투영(메모리 그래프)을 모두 정리: 이 셀은 실행만 하세요.
# GDS 투영은 데이터베이스가 아니라 메모리에 올린 분석용 사본입니다. 앞 실습의 사본이 남아 있으면
# 같은 이름으로 다시 투영할 때 충돌하므로, 시작할 때 목록을 확인해 전부 내려 둡니다.
# 목록을 먼저 받아 두고 한 개씩 내린다
for row in run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName"):
    run_cypher("CALL gds.graph.drop($g) YIELD graphName", g=row["graphName"])
print("남은 투영:", run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName"))

In [ ]:
# [제공 코드] 아시아 항공 노선망 적재: 이 셀은 실행만 하세요(실습에 쓸 그래프를 만듭니다).
import pandas as pd

# 1) CSV 두 벌을 읽는다. 공항 767행, 노선 8,045행
airports = pd.read_csv("data/airports_asia_nodes.csv")   # iata, name, city, country, lat, lon
routes = pd.read_csv("data/airports_asia_edges.csv")     # from_iata, to_iata, km, hours, airlines

# 2) iata(공항 코드 세 글자)로 공항을 찾아 관계를 잇습니다. 인덱스가 없으면 한 줄마다 전체를 훑으므로
# 31일차에서 배운 유일성 제약(인덱스가 함께 생깁니다)을 먼저 겁니다.
run_cypher("CREATE CONSTRAINT airport_iata IF NOT EXISTS FOR (a:Airport) REQUIRE a.iata IS UNIQUE")
run_cypher("""UNWIND $rows AS r
    CREATE (:Airport {iata: r.iata, name: r.name, city: r.city, country: r.country,
                      lat: r.lat, lon: r.lon})""", rows=airports.to_dict("records"))

# 3) 노선은 방향이 있습니다. 왕복이면 두 행(A→B, B→A), 원본에 한 방향만 있으면 한 행입니다.
# km 는 두 공항 사이 대권 거리, hours 는 km/800 + 1 (순항 800km/h 에 이착륙·환승 1시간을 더한,
# 교재가 정한 값), airlines 는 그 방향을 운항하는 항공사 수입니다.
run_cypher("""UNWIND $rows AS r
    MATCH (a:Airport {iata: r.from_iata}), (b:Airport {iata: r.to_iata})
    CREATE (a)-[:ROUTE {km: r.km, hours: r.hours, airlines: r.airlines}]->(b)""",
           rows=routes.to_dict("records"))

print("공항:", run_cypher("MATCH (a:Airport) RETURN count(a) AS n")[0]["n"],
      "· 노선 관계(방향별):", run_cypher("MATCH (:Airport)-[r:ROUTE]->() RETURN count(r) AS n")[0]["n"])

In [ ]:
# [제공 코드] 아시아 항공 노선망을 무방향으로 투영합니다: 이 셀은 실행만 하세요.
# 노선은 방향이 있지만 '어느 공항끼리 이어져 있는가'에는 방향이 의미가 없어 UNDIRECTED 로 만듭니다.
run_cypher("CALL gds.graph.drop('air', false) YIELD graphName")
stats = run_cypher('''
    CALL gds.graph.project('air', 'Airport',
      {ROUTE: {orientation: 'UNDIRECTED', properties: ['km', 'hours', 'airlines']}})
    YIELD graphName, nodeCount, relationshipCount
    RETURN graphName, nodeCount, relationshipCount''')
print(stats[0])

In [ ]:
# [제공 코드] 다섯 부서 부분망 적재: 이 셀은 실행만 하세요(실습에 쓸 그래프를 만듭니다).
import pandas as pd

# 시연용 :Airport 와 레이블도 관계 이름도 달라서 서로 섞이지 않는다
people = pd.read_csv("data/email_dept_group2_nodes.csv")   # employee_id, dept
mails = pd.read_csv("data/email_dept_group2_edges.csv")     # from_id, to_id

# employee_id 로 사람을 찾아 관계를 잇습니다. 인덱스가 없으면 한 줄마다 전원을 훑으므로
# 31일차에서 배운 유일성 제약(인덱스가 함께 생깁니다)을 먼저 겁니다.
run_cypher("CREATE CONSTRAINT member_id IF NOT EXISTS FOR (p:Member) REQUIRE p.employee_id IS UNIQUE")
run_cypher("UNWIND $rows AS r CREATE (:Member {employee_id: r.employee_id, dept: r.dept})",
           rows=people.to_dict("records"))

# 메일은 방향이 있습니다. 서로 주고받은 사이면 두 관계(a→b, b→a)가 생깁니다.
# 항공의 왕복 노선과 같은 모양이라, 이 그래프도 뒤에서 무방향으로 투영한다
run_cypher("""UNWIND $rows AS r
    MATCH (a:Member {employee_id: r.from_id}), (b:Member {employee_id: r.to_id})
    CREATE (a)-[:MAILED]->(b)""", rows=mails.to_dict("records"))

print("구성원:", run_cypher("MATCH (p:Member) RETURN count(p) AS n")[0]["n"],
      "· 메일 관계:", run_cypher("MATCH (:Member)-[r:MAILED]->() RETURN count(r) AS n")[0]["n"])

오늘 단원(교안 두 권과 과제 두 권)이 쓰는 그래프 **다섯**을 먼저 봅니다. 시연·따라하기·과제가 서로 다른 그래프를 쓰므로, 지금 보는 것이 어느 그래프인지 늘 확인하며 따라오세요.

<img src="images/교안/오늘의_그래프_다섯.png" width="900">

이 노트북은 **데이터 두 벌**을 씁니다.

- **강사 시연**: 아시아 항공 노선망(`:Airport`, 공항 767곳, 국가·지역 47개). 새 개념은 여기서 보여 드립니다.
- **함께 따라하기**: 사내 메일 다섯 부서 부분망(`:Member`, 194명, 부서 5개). 배운 것을 **다른 그래프에** 직접 적용해 봅니다. 1절에서 만납니다.

두 그래프는 레이블도 관계 이름도 달라 서로 건드리지 않습니다. 위 셀을 한 번씩만 실행하면 됩니다.

<img src="images/교안/항공_아시아망_구성.png" width="900">

노선 자료는 **2014년 6월에 갱신이 멈춘 역사 자료**입니다. 지금 운항 여부와 다를 수 있고, 시간표가 아니라 **노선이 있었는가**만 담겨 있습니다. `country` 열은 원본(OpenFlights) 표기라 대만·홍콩·마카오가 따로 들어 있습니다. 그래서 이 교재는 이 열을 **국가·지역**이라고 부릅니다.

---
## 데이터 살펴보기

새 데이터는 분석하기 전에 **먼저 규모와 모양을 봅니다**. 공항이 몇 곳이고, 노선 관계가 몇 개이며, 국가·지역이 몇 개인지부터 확인합니다.

In [ ]:
# 공항 수·국가 가짓수·노선 관계 수를 한 쿼리로 함께 센다
# COUNT { } 는 패턴 개수를 세는 문법. 노선은 방향이 있으므로 방향 그대로 센다
size_rows = run_cypher('''
    MATCH (a:Airport)
    RETURN count(a) AS airports,
           count(DISTINCT a.country) AS countries,
           COUNT { ()-[:ROUTE]->() } AS routes''')
print(size_rows[0])

> 노선 관계가 **8,045개**로 나옵니다. 이 수는 **방향별**입니다. 인천에서 도쿄로 가는 노선과 도쿄에서 인천으로 오는 노선이 **각각 한 개**입니다. 그러면 실제 노선은 몇 개일까요. 다음 셀에서 **쌍**으로 세어 봅니다.

In [ ]:
# 노선을 '쌍'으로 세어 본다. 왕복이면 관계가 둘이고, 한 방향만 있으면 하나다
# COUNT { (b)-[:ROUTE]->(a) } > 0 이 '반대 방향 노선이 있는가' 를 묻는다
shape = run_cypher('''
    MATCH (a:Airport)-[:ROUTE]->(b:Airport)
    WITH COUNT { (b)-[:ROUTE]->(a) } > 0 AS round_trip
    WITH count(*) AS directed,
         sum(CASE WHEN round_trip THEN 1 ELSE 0 END) / 2 AS round_trip_pairs,
         sum(CASE WHEN round_trip THEN 0 ELSE 1 END) AS one_way_pairs
    RETURN directed, round_trip_pairs + one_way_pairs AS pairs,
           round_trip_pairs, one_way_pairs''')
print(shape[0])

> 방향별 8,045건은 쌍으로 세면 **4,070쌍**입니다. 그중 **3,975쌍은 왕복**(관계 둘)이고 **95쌍은 한 방향뿐**입니다(예: ADE→RUH, AGR→HJR).

한 방향뿐인 쌍을 편도라고 부르겠습니다. 다만 이것은 **원본에 한 방향만 들어 있다**는 뜻이지 실제로 편도 운항이라는 보장은 아닙니다. 수집 과정에서 빠졌을 수도 있습니다. **데이터가 말하는 것과 세상이 그렇다는 것은 다릅니다.** 이 차이는 1-1 에서 차수를 읽을 때 곧바로 다시 나옵니다.

In [ ]:
# 노선이 하나도 없는 공항이 있는지 센다. 있으면 뒤에서 노드 하나짜리 묶음으로 나온다
alone = run_cypher("MATCH (a:Airport) WHERE COUNT { (a)--() } = 0 RETURN count(a) AS alone")
print('노선이 하나도 없는 공항:', alone[0]['alone'], '곳')

> **한 곳도 없습니다.** 이 데이터는 서로 이어진 가장 큰 덩어리만 잘라 낸 것이기 때문입니다. 고립된 공항이 없다고 알고리즘이 작은 묶음을 못 만드는 것은 아니지만, 실측 120회에서 공항 하나짜리 묶음은 한 번도 나오지 않았습니다. 작은 묶음이 나온다면 그것은 고립된 점이 아니라 **바깥으로 노선이 적게 뻗는 지역**입니다.

In [ ]:
# 국가·지역마다 공항이 몇 곳인지 세어 많은 순으로 정렬한다
import pandas as pd

country_rows = run_cypher('''
    MATCH (a:Airport)
    RETURN a.country AS country, count(*) AS airports
    ORDER BY airports DESC''')
country_size = pd.DataFrame(country_rows)   # 열은 country, airports 두 개
print('국가·지역 수:', len(country_size),
      '· 가장 많은 곳:', country_size['airports'].max(),
      '· 가장 적은 곳:', country_size['airports'].min())

In [ ]:
display(country_size.head(8))

> 중국이 167곳으로 가장 많고 인도(67곳)·일본(61곳)·인도네시아(57곳)·러시아(48곳)가 뒤를 잇습니다. 국가·지역은 47개인데 공항은 **상위 몇 곳에 몰려 있습니다.** 이 쏠림은 3절에서 교차표를 읽을 때 다시 나옵니다. 큰 나라 하나가 묶음 하나를 거의 채웁니다.

In [ ]:
# 노선마다 몇 개 항공사가 다니는지 센다. 4절에서 '노선 굵기'로 쓸 값이다
airline_rows = run_cypher('''
    MATCH ()-[r:ROUTE]->()
    RETURN r.airlines AS airlines, count(*) AS routes
    ORDER BY airlines''')
display(pd.DataFrame(airline_rows).head(8))

> 관계에는 숫자가 셋 붙어 있습니다.

- **`airlines`**: 그 방향을 운항하는 **항공사 수**입니다(코드셰어 포함). 항공사가 하나뿐인 노선이 3,805건으로 가장 많고 평균은 2.17개입니다. 가장 굵은 노선은 푸껫(HKT)과 방콕(BKK) 사이로 **13개** 항공사가 다닙니다. 이 값을 4-2 에서 커뮤니티 탐지에 넘겨 봅니다.
- **`km`**: 두 공항 사이 대권 거리입니다(41km 에서 8,645km, 중앙값 1,037km).
- **`hours`**: `km/800 + 1` 로 **교재가 정한 값**입니다(순항 800km/h 에 이착륙·환승 1시간). 데이터에 원래 있던 값이 아닙니다.

`km` 와 `hours` 는 **다음 교시**에서 최단 경로의 잣대로 씁니다. 오늘은 `airlines` 만 씁니다.

---
# 1. 커뮤니티란 무엇이고, 왜 무방향으로 투영할까

<img src="images/교안/커뮤니티란_촘촘한_묶음.png" width="820">

안쪽으로는 촘촘하고 바깥으로는 성긴 묶음, 이것이 **커뮤니티**입니다. 항공 노선망에서는 눈으로도 짐작이 됩니다. 같은 권역 안의 공항끼리는 노선이 빽빽하고, 권역과 권역 사이는 큰 공항 몇 곳을 거쳐 가늘게 이어집니다. 커뮤니티 탐지는 **나라 이름을 주지 않아도** 연결의 촘촘함만으로 이 묶음을 찾아냅니다.

## 1-1. 노선망을 무방향으로 투영하기

### 왜 필요할까요?
PageRank 는 공항 하나의 중요도를 봤습니다. 하지만 노선망을 이해하려면 "**어느 공항끼리 한 덩어리처럼 이어져 있는가**"를 봐야 할 때가 많습니다. 그 답이 커뮤니티입니다.

### 문법: 무방향(UNDIRECTED) 투영
우리 데이터의 관계는 `(a)-[:ROUTE]->(b)`, 즉 "a 에서 b 로 가는 노선이 있다"라 방향이 있습니다. 하지만 "**어느 공항끼리 이어져 있는가**"를 볼 때는 어느 쪽으로 가는지가 중요하지 않습니다. 그래서 투영할 때 `orientation: 'UNDIRECTED'` 를 줍니다.

이건 권장 사항이 아니라 **필수**입니다. 방향이 있는 투영을 Leiden 에 넘기면 결과가 나빠지는 게 아니라 **아예 실행을 거부**합니다(`Leiden requires relationship projections to be UNDIRECTED`). 그런 에러를 만나면 알고리즘이 아니라 **투영을 고쳐야** 합니다.

관계 속성 `km`·`hours`·`airlines` 도 함께 싣습니다. 투영은 **그 순간의 사본**이라, 나중에 필요한 값은 만들 때 실어 두어야 합니다. `airlines` 는 4-2 에서 씁니다.

In [ ]:
# 아시아 항공 노선망을 무방향으로 투영합니다.
# 노선은 방향이 있지만 '어느 공항끼리 이어져 있는가'에는 방향이 의미가 없어 UNDIRECTED 로 만듭니다.
# drop 의 두 번째 인자 false 는 '없으면 그냥 넘어가라'는 뜻이다.
# 준비 셀에서 만든 같은 이름의 투영을 내리고 여기서 다시 만든다
run_cypher("CALL gds.graph.drop('air', false) YIELD graphName")
stats = run_cypher('''
    CALL gds.graph.project('air', 'Airport',
      {ROUTE: {orientation: 'UNDIRECTED', properties: ['km', 'hours', 'airlines']}})
    YIELD graphName, nodeCount, relationshipCount
    RETURN graphName, nodeCount, relationshipCount''')
print(stats[0])

> `relationshipCount` 가 **16,090** 으로 나옵니다. 적재한 관계 8,045개의 정확히 **두 배**입니다. 무방향 투영은 관계 하나를 **양쪽 방향 모두**(a 에서 b, b 에서 a) 저장하기 때문입니다. 정상입니다.

투영이 정말 무방향인지 확인하는 방법이 하나 더 있습니다. `gds.graph.list` 의 `degreeDistribution` 을 보는 것입니다.

<img src="images/교안/무방향_투영과_차수.png" width="860">

관계 몇 개짜리 작은 예로 보면 두 배가 되는 이유가 한눈에 들어옵니다. 그리고 여기서 **한 공항의 차수와 그 공항의 상대 수가 다른 값**이라는 것도 함께 봐 두세요. 왕복 노선은 관계가 둘이라 차수를 둘로 세지만, 상대 공항은 한 곳입니다. **다음 교시(교안_02) 1절**에서 `nodeSimilarity` 의 `degreeCutoff` 를 정할 때 이 차이가 다시 나옵니다.

In [ ]:
# 투영이 무방향인지 확인: 방향이면 차수가 절반 수준으로 준다
dist = run_cypher("CALL gds.graph.list('air') YIELD degreeDistribution "
                  "RETURN degreeDistribution")
print(dist[0]['degreeDistribution'])

> `p50` 이 **6** 으로 나옵니다. 여기서 한 번 멈춰야 합니다. **이 값은 상대 공항 수가 아니라 관계 수**입니다. 바로 위에서 확인했듯 무방향 투영은 왕복 노선을 관계 두 개로 세기 때문입니다. **서로 다른 상대 공항이 몇 곳인지**로 세면 중앙값은 **4곳**입니다. 두 배 관계는 공항 하나하나에서 성립하는 것이지(왕복 노선만 있는 공항은 차수가 상대 수의 정확히 두 배) 중앙값끼리 성립하는 것은 아닙니다. 두 중앙값은 서로 다른 공항에서 잡히고 편도 노선도 섞여 있기 때문입니다. 아래 셀에서 직접 세어 봅니다.

In [ ]:
# 관계 수(COUNT)와 서로 다른 상대 공항 수(DISTINCT)를 나란히 센다
# 다음 교시의 degreeCutoff 는 '상대 수' 기준이라 어느 쪽인지 구분해야 한다
spread = run_cypher('''
    MATCH (a:Airport)
    OPTIONAL MATCH (a)--(x:Airport)
    WITH a, COUNT { (a)--() } AS rels, count(DISTINCT x) AS partners
    RETURN percentileCont(rels, 0.5) AS median_rels,
           percentileCont(partners, 0.5) AS median_partners''')
print(spread[0])

In [ ]:
# 차수가 큰 공항 다섯 곳. 허브일수록 두 값이 크게 벌어진다
hub_rows = run_cypher('''
    MATCH (a:Airport)--(x:Airport)
    WITH a, count(*) AS degree, count(DISTINCT x) AS partners
    RETURN a.iata AS iata, a.city AS city, degree, partners
    ORDER BY degree DESC LIMIT 5''')
display(pd.DataFrame(hub_rows))

> 허브인 베이징(PEK)은 차수가 **313** 인데 실제 상대 공항은 **157곳**입니다. 왕복이면 관계가 둘이니 차수는 상대 수의 두 배에 가깝습니다. 다만 정확히 314(157 x 2)가 아니라 313 인데, 상대 가운데 1곳과는 **한 방향 노선만** 있기 때문입니다. 데이터 살펴보기에서 센 편도 95쌍이 여기 이렇게 나타납니다.

**차수를 상대 수로 읽으면 두 배로 부풀려 보고하게 됩니다.** 무방향 투영의 숫자를 인용할 때는 그 값이 관계 수인지 상대 수인지 먼저 확인하십시오.

### 🖐️ 함께 따라하기: 다섯 부서 부분망을 무방향으로 투영하기

<img src="images/교안/메일_다섯부서_구성.png" width="820">

따라하기는 **시연과 다른 그래프**로 합니다. 사내 메일망에서 다섯 부서만 잘라 낸 부분망입니다. 메일도 노선처럼 방향이 있고(`a` 가 `b` 에게 보냈다), 서로 주고받았으면 관계가 둘입니다. "누가 누구와 함께 일하는가"를 볼 때는 방향이 필요 없으니 여기서도 무방향으로 투영합니다.

`:Member` 와 `MAILED` 로 **투영 이름 `'team'`** 을 만드세요. `orientation: 'UNDIRECTED'` 를 주고, 만든 뒤 `nodeCount`·`relationshipCount` 를 출력해 확인합니다. 이 데이터의 관계에는 숫자가 붙어 있지 않으므로 속성은 싣지 않습니다.

**확인 기준**: `nodeCount` 는 **194**, `relationshipCount` 는 **4,878**(적재한 2,439개의 두 배)입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 같은 이름의 투영이 남아 있을 수 있으니 gds.graph.drop('team', false) 로 먼저 내린다
# 2) gds.graph.project 로 'team' 을 만든다. 레이블은 'Member', 관계는 MAILED,
#    관계 설정에 orientation: 'UNDIRECTED' 를 준다
# 3) YIELD graphName, nodeCount, relationshipCount 를 받아 출력한다

### ✅ 바로 확인 퀴즈

**1.** 노선망을 `UNDIRECTED` 로 투영하는 이유는?

<details><summary>정답 보기</summary>

"어느 쪽으로 가는 노선인가"가 아니라 "**서로 이어져 있는가**"를 보려는 것이기 때문입니다. 게다가 Leiden 은 방향 투영을 아예 거부합니다.

</details>

**2.** 적재한 관계는 8,045개인데 투영의 `relationshipCount` 는 16,090 입니다. 버그일까요?

<details><summary>정답 보기</summary>

아닙니다. 무방향 투영은 관계 하나를 양쪽 방향으로 각각 저장해서 정확히 두 배가 됩니다.

</details>

**3.** 베이징(PEK)의 차수는 313 인데 상대 공항은 157곳입니다. 왜 정확히 두 배(314)가 아닐까요?

<details><summary>정답 보기</summary>

상대 가운데 1곳과는 원본에 **한 방향 노선만** 있기 때문입니다. 왕복이면 관계가 둘이라 차수를 둘 올리지만, 한 방향뿐이면 하나만 올립니다.

</details>

---
# 2. Leiden 으로 커뮤니티 찾기

**Leiden** 은 지금 가장 널리 쓰이는 커뮤니티 탐지 알고리즘입니다. 그래서 이 단원의 기준 알고리즘으로 씁니다.

## 2-1. 커뮤니티를 찾고 크기 분포로 먼저 훑기

### 왜 필요할까요?
Leiden 은 오래 쓰인 Louvain 을 개선한 것으로, Louvain 이 가끔 만들어 내던 **속이 끊긴 커뮤니티**(같은 묶음으로 묶였는데 그 안에서 서로 이어지지 않는 경우)가 생기지 않도록 보장합니다.

### 문법: `gds.leiden.stream`
```cypher
CALL gds.leiden.stream('투영이름')
YIELD nodeId, communityId
```
- `nodeId` 는 GDS 내부 번호라 그대로는 못 읽습니다. `gds.util.asNode(nodeId)` 로 실제 노드를 꺼냅니다.
- `communityId` 는 **같은 묶음에 같은 번호**가 붙었다는 뜻일 뿐, 번호 자체에 의미는 없습니다.

In [ ]:
# stream 은 결과만 돌려주고 그래프에는 저장하지 않는다(저장은 뒤의 write)
# lat·lon 도 함께 받아 둔다. 6절에서 이 결과를 그대로 지도에 찍는다
comm_rows = run_cypher('''
    CALL gds.leiden.stream('air')
    YIELD nodeId, communityId
    RETURN gds.util.asNode(nodeId).iata AS iata,
           gds.util.asNode(nodeId).city AS city,
           gds.util.asNode(nodeId).country AS country,
           gds.util.asNode(nodeId).lat AS lat,
           gds.util.asNode(nodeId).lon AS lon,
           communityId AS community''')
community = pd.DataFrame(comm_rows)   # 열은 iata, city, country, lat, lon, community
print('찾은 묶음 수:', community['community'].nunique())

In [ ]:
display(community.head())

묶음이 몇 개인지 알았으니 **크기 분포**를 봅니다. 개수만 보고 넘어가면 결과를 크게 오해합니다.

In [ ]:
# 묶음별 공항 수. 큰 것부터 늘어놓아야 쏠림이 보인다
sizes = community['community'].value_counts()   # 묶음 번호 -> 공항 수. 많은 순으로 정렬돼 나온다
print('묶음 수:', len(sizes))
print('크기(큰 순서):', sorted(sizes, reverse=True))

In [ ]:
# 가장 작은 묶음에는 어느 국가·지역이 들어 있는지 함께 본다
smallest = sizes.index[-1]                      # value_counts 는 적은 쪽이 뒤에 온다
peek = community[community['community'] == smallest]
print('가장 작은 묶음:', len(peek), '곳')

In [ ]:
print(peek['country'].value_counts().head())

> 묶음은 여러 번 돌려 보면 **5개에서 8개** 사이가 나옵니다(가장 자주 나온 것은 7개). 가장 큰 묶음은 **170곳에서 211곳** 사이입니다. 공항이 767곳이니 **가장 큰 묶음 하나가 전체의 22%에서 28%** 를 차지합니다.

커뮤니티 탐지 결과를 받으면 **개수보다 크기 분포를 먼저 보십시오.** 한쪽으로 몰렸는지, 작은 묶음이 몇 개인지가 결과의 쓸모를 좌우합니다.

가장 작은 묶음에 어느 국가·지역이 들어 있는지도 함께 찍었습니다. **크기가 작다고 찌꺼기가 아닙니다.** 바깥으로 노선이 적게 뻗는 쪽이 따로 묶인 것이고, 그 정체는 3절 교차표에서 이름으로 확인합니다.

> 실행할 때마다 **어느 묶음에 어느 공항이 들어가는지**가 조금씩 달라집니다. 묶음 수는 거의 그대로인데도 그렇습니다. 왜 그런지는 5절에서 다룹니다.

## 2-2. 결과를 그래프에 저장하기: `write`
`stream` 은 결과를 화면으로 돌려줄 뿐 그래프에는 아무것도 남기지 않습니다. 뒤에서 일반 Cypher 로 "같은 묶음의 공항 목록" 같은 조회를 하려면 **노드 속성으로 저장**해야 합니다. `write` 를 씁니다.

In [ ]:
# 묶음 번호를 노드 속성 community 로 저장한다. writeProperty 에 준 이름이 곧 속성 이름이다
# 돌려주는 communityCount 와 modularity 는 알고리즘 결과라 실행마다 조금씩 달라진다
written = run_cypher('''
    CALL gds.leiden.write('air', { writeProperty: 'community' })
    YIELD communityCount, nodePropertiesWritten, modularity
    RETURN communityCount, nodePropertiesWritten, round(modularity, 4) AS modularity''')
print(written[0])

In [ ]:
# 저장했으니 이제 GDS 없이 평범한 Cypher 로도 조회된다
saved = run_cypher('''
    MATCH (a:Airport)
    WHERE a.community IS NOT NULL
    RETURN a.community AS community, count(*) AS airports
    ORDER BY airports DESC LIMIT 5''')
print(saved)

> 함께 나온 `modularity` 는 "이 분할이 얼마나 무리다운가"를 재는 점수입니다. 4절에서 이 점수로 알고리즘을 견줍니다.

**여기서 저장한 `community` 는 2-1 의 `stream` 과 다른 실행입니다.** 두 번 돌렸으니 번호도 구성도 조금 다릅니다. 뒤에서 어느 쪽을 쓰는지 그때그때 밝히겠습니다.

### 🖐️ 함께 따라하기: 다섯 부서 부분망의 묶음 수와 크기

`'team'` 투영에 Leiden 을 돌려 묶음을 찾고, **묶음 수**와 **크기 목록**(큰 것부터)을 출력하세요. 결과는 pandas 로 받아도 되고 Cypher 집계로 세도 됩니다.

**확인 기준**: 묶음은 **5개**, 크기는 큰 순서대로 **51·45·38·32·28명** 입니다. 이 그래프는 항공망과 달리 **여러 번 돌려도 늘 같은 답**이 나옵니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) gds.leiden.stream('team') 을 YIELD nodeId, communityId 로 받는다
# 2) employee_id·dept·communityId 세 열을 돌려받아 DataFrame 으로 만든다
#    (이름은 team_community 로 둔다. 3절·6절 따라하기가 이 이름과 dept 열을 그대로 쓴다)
# 3) 묶음 수(nunique)와 크기 목록(value_counts 를 내림차순 리스트로)을 출력한다

### ✅ 바로 확인 퀴즈

**1.** `stream` 과 `write` 의 차이는?

<details><summary>정답 보기</summary>

`stream` 은 결과를 그때그때 돌려주기만 하고 그래프에는 남기지 않습니다. `write` 는 각 노드에 속성으로 저장해서, 이후 평범한 Cypher 조회에서도 쓸 수 있게 만듭니다.

</details>

**2.** 동료가 "묶음이 7개 나왔습니다"라고만 보고했습니다. 무엇을 더 물어야 할까요?

<details><summary>정답 보기</summary>

**크기 분포**입니다. 그중 하나가 공항 170곳을 넘게 담고 있다면(전체의 22% 이상) "묶음이 여럿"이라는 말과 실제 모양이 많이 다릅니다. 개수만으로는 결과가 쓸모 있는지 판단할 수 없습니다.

</details>

---
# 3. 찾아낸 묶음이 실제 지도와 얼마나 맞는가

세 단계로 봅니다. **눈으로 보고**(교차표), **직접 세고**(순도), **표준 지표로 요약**합니다(NMI).

<img src="images/교안/커뮤니티_정답_교차표.png" width="860">

세 단계가 어떻게 이어지는지 먼저 그림으로 봐 두세요. 교차표 한 장에서 순도와 NMI 가 모두 나옵니다.

## 3-1. 교차표로 눈으로 본다

### 왜 필요할까요?
커뮤니티 탐지는 **정답을 주지 않아도 무리를 만들어 냅니다.** 뒤집어 말하면 **아무 그래프에나 돌려도 무언가는 나옵니다.** 그 결과가 진짜 의미 있는 묶음인지, 아니면 알고리즘이 그냥 선을 그은 것인지는 별도로 확인해야 합니다.

우리 데이터에는 **정답이 있습니다.** 공항 767곳이 각각 어느 국가·지역인지 `country` 속성에 들어 있습니다. 알고리즘은 이 값을 전혀 보지 않고 노선 연결만으로 묶음을 만들었으니, 둘을 맞대 보면 "**노선만으로 지도를 어느 정도 복원했는가**"를 잴 수 있습니다.

묶음을 행, 국가·지역을 열로 놓고 공항 수를 세면 어디가 맞고 어디가 어긋났는지 한눈에 보입니다. 33일차에서 쓴 `pd.crosstab` 그대로입니다.

In [ ]:
# 묶음(행) x 국가·지역(열) 공항 수 교차표. 2-1 의 stream 결과를 쓴다
cross = pd.crosstab(community['community'], community['country'])

# 국가·지역 47개를 다 담으면 읽을 수 없으니 공항이 많은 10곳만 남긴다
top_countries = cross.sum().sort_values(ascending=False).head(10).index
display(cross[top_countries].sort_values(by=list(top_countries), ascending=False))

<img src="images/교안/커뮤니티_국가_교차표.png" width="820">

같은 표를 색으로 칠하면 이렇게 보입니다. 색이 진할수록 그 칸의 공항이 많습니다. **한 줄에 진한 칸이 하나뿐이면** 그 묶음은 한 국가·지역으로만 이뤄진 것이고, **진한 칸이 여러 개면** 여러 나라가 섞인 것입니다.

> 묶음 번호와 줄 순서는 실행할 때마다 달라집니다. **모양**을 보세요.

## 3-2. 순도를 읽는다

**순도**(purity)는 한 묶음에서 **가장 많은 국가·지역이 차지하는 비율**입니다. 1.0 이면 그 묶음이 한 나라로만 이뤄졌다는 뜻입니다. 묶음마다 따로 계산합니다.

In [ ]:
# 묶음마다 공항이 가장 많은 국가·지역과 그 수만 뽑아 표로 만든다
profile = pd.DataFrame({
    'airports': cross.sum(axis=1),
    'top_country': cross.idxmax(axis=1),
    'from_top': cross.max(axis=1),
})
# 순도 = 가장 많은 국가·지역이 차지하는 비율. 1.0 이면 한 나라로만 구성
profile['purity'] = (profile['from_top'] / profile['airports']).round(3)
display(profile.sort_values('purity', ascending=False))

> 순도가 가장 높은 묶음은 **0.92에서 1.0** 사이로 나옵니다. 알고리즘이 나라 이름을 보지도 않고 **거의 한 나라만 골라 담은** 것입니다. 공항이 많은 큰 나라는 자기들끼리 노선이 촘촘해서 그 자체로 한 묶음이 됩니다.

반대로 순도가 가장 낮은 묶음은 **0.3에서 0.39** 입니다. 여러 나라 공항이 뒤섞여 있다는 뜻입니다.

**섞인 묶음은 알고리즘이 틀린 것일까요?** 그렇게 단정할 수 없습니다. 나라가 다른 공항들이 한 묶음이 되었다는 것은 **그 공항들 사이에 노선이 실제로 촘촘하다**는 뜻입니다. 국경이 아니라 **권역**으로 묶인 것입니다. 커뮤니티 탐지의 쓸모가 바로 여기 있습니다. **행정 구역과 실제 왕래의 차이**를 데이터로 보여 줍니다.

In [ ]:
# 가장 섞인 묶음을 하나 골라, 그 안에 어느 국가·지역 공항이 몇 곳씩 있는지 본다
mixed_id = profile['purity'].idxmin()          # 순도가 가장 낮은 묶음
mixed = cross.loc[mixed_id]                    # 그 묶음 한 줄(국가·지역별 공항 수)
print(f'묶음 {mixed_id} 는 공항 {int(mixed.sum())}곳이고, 국가·지역이 {int((mixed > 0).sum())}개 섞여 있다')
print(mixed[mixed > 0].sort_values(ascending=False).head(6))

## 3-3. NMI 로 전체를 한 숫자로 요약한다

묶음마다 순도를 따로 보는 것은 정확하지만, 여러 설정을 비교할 때는 **전체를 한 숫자로** 요약해야 편합니다. 표준 지표가 **NMI**(정규화 상호정보량)입니다.

| NMI 값 | 뜻 |
|---|---|
| 1.0 | 두 나눔이 완전히 같다(묶음 = 국가·지역) |
| 0.7 ~ 0.9 | 대체로 맞고 일부만 어긋난다 |
| 0.4 ~ 0.6 | 절반쯤 맞는다. 겹치는 구조가 있다 |
| 0.0 | 두 나눔이 아무 관계가 없다 |

**번호가 달라도 상관없습니다.** NMI 는 "묶음 3번 = 일본"처럼 이름을 맞추는 게 아니라 "**같은 묶음인 두 공항이 같은 나라이기도 한가**"를 봅니다. 그래서 실행마다 번호가 바뀌어도 값은 안정적입니다.

In [ ]:
# NMI 는 sklearn 에 이미 들어 있다. 두 라벨 목록을 같은 순서로 넘기면 된다
from sklearn.metrics import normalized_mutual_info_score

# 공항 전부를 넣는다. 일부를 빼면 쉬운 부분만 골라 낸 셈이라 점수가 부풀려진다
nmi = normalized_mutual_info_score(community['country'], community['community'])
print(f'묶음 대 국가·지역 NMI: {nmi:.4f}')

> **0.738 안팎**이 나옵니다(여러 번 돌려 보면 0.675에서 0.767 사이입니다). 위 표로 읽으면 "**대체로 맞고 일부만 어긋난다**"입니다.

이 값을 어떻게 읽어야 할까요. 1.0 이 아니라고 실패가 아닙니다. 애초에 국경과 노선망이 같을 이유가 없습니다. 오히려 이 데이터가 말하는 것은 이렇습니다.

- 어떤 묶음은 **거의 한 나라로만** 채워집니다(순도 0.92 이상). 그 나라 안에서만 노선이 촘촘하다는 뜻입니다.
- 어떤 묶음은 **여러 나라가 섞여** 있습니다. 국경을 가로지르는 왕래가 많다는 뜻입니다.
- 국가·지역은 47개인데 묶음은 **5개에서 8개**뿐입니다. **알고리즘이 찾은 것은 나라가 아니라 권역**이라는 뜻이고, 5절에서 해상도 값을 올려 나라 쪽으로 쪼개 봅니다.

실무에서 이 검증을 건너뛰면 어떻게 될까요. "묶음 여덟 개를 찾았습니다"라고만 보고하게 되고, 그것이 지도를 그대로 베낀 것인지 새로운 사실인지 아무도 모릅니다. **정답이 있으면 반드시 재십시오.**

### 🖐️ 함께 따라하기: 다섯 부서 부분망도 정답과 맞춰 보기

앞에서 만든 `team_community` 로 같은 검증을 하세요. 이 그래프의 정답은 **부서**(`dept`)입니다.

1. `pd.crosstab` 으로 묶음(행) 대 부서(열) 교차표를 만들어 출력합니다.
2. `normalized_mutual_info_score` 로 NMI 를 구해 소수 넷째 자리까지 출력합니다.

**확인 기준**: NMI 가 **0.963** 로 나옵니다. 항공 노선망의 0.738 보다 훨씬 높습니다. **부서가 5개뿐이고 메일이 부서 안에서 주로 오가서** 그렇습니다. 같은 알고리즘이라도 **그래프가 달라지면 성적이 달라진다**는 것을 여기서 확인하세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) team_community 의 커뮤니티 열과 부서 열로 pd.crosstab 교차표를 만들어 display 한다
# 2) 부서와 커뮤니티를 normalized_mutual_info_score 에 넘겨 NMI 를 구한다
# 3) f-string 으로 소수 넷째 자리까지 출력한다

### ✅ 바로 확인 퀴즈

**1.** 같은 알고리즘인데 항공 노선망에서는 NMI 가 0.738, 다섯 부서 부분망에서는 0.963 이 나왔습니다. 알고리즘 성능이 달라진 걸까요?

<details><summary>정답 보기</summary>

아닙니다. **그래프가 다릅니다.** 부서 다섯 개짜리 부분망은 묶음이 뚜렷하게 갈려 있고, 국가·지역 47개가 얽힌 노선망은 국경을 가로지르는 노선이 많습니다. 지표는 알고리즘만이 아니라 **데이터의 성질**을 함께 반영합니다.

</details>

**2.** 순도가 0.3 인 묶음을 보고 "알고리즘이 틀렸다"고 결론 내려도 될까요?

<details><summary>정답 보기</summary>

안 됩니다. 나라가 다른 공항들이 한 묶음이 되었다는 것은 그 사이에 노선이 촘촘하다는 뜻입니다. **국경이 아니라 권역을 찾아낸 것**일 수 있습니다. 그 묶음에 어느 나라들이 들어 있는지 확인해야 판단할 수 있습니다.

</details>

**3.** 실행할 때마다 묶음 번호가 바뀌는데 NMI 값은 거의 그대로입니다. 왜일까요?

<details><summary>정답 보기</summary>

NMI 는 번호를 맞대는 게 아니라 "**같이 묶였는가 아닌가**"만 봅니다. 번호를 통째로 바꿔도 묶음이 같으면 값이 같습니다.

</details>

---
# 4. Louvain·라벨 전파와 나란히 놓고 고르기

## 4-1. 세 알고리즘을 같은 잣대로 재기

### 왜 필요할까요?
커뮤니티 탐지 알고리즘은 여럿입니다. "어느 것이 제일 좋은가"라는 질문에는 답이 없고, **이 데이터에서 어느 것이 나은가**를 재서 골라야 합니다.

| 알고리즘 | 성격 |
|---|---|
| **Leiden** | Louvain 개선판. 속이 끊긴 묶음이 생기지 않도록 보장. 묶음 수를 `gamma` 로 조절 |
| **Louvain** | 오래 쓰여 자료가 많다. 빠르고 대체로 좋다 |
| **라벨 전파(Label Propagation)** | 이웃의 다수결로 라벨을 퍼뜨린다. 가장 빠르지만 결과가 불안정하다 |

### 무엇으로 비교하나: 모듈러리티
**모듈러리티**는 "이 분할이 얼마나 무리다운가"를 재는 점수입니다. **묶음 안쪽의 관계가 무작위로 이었을 때보다 얼마나 더 많은가**를 봅니다.

- 0.3 ~ 0.7 이면 뚜렷한 무리 구조가 있다고 봅니다.
- 0 에 가까우면 **무리라고 부를 게 없다**는 뜻입니다.
- 정답이 없어도 잴 수 있어서, 실무에서는 이 점수로 설정을 고릅니다.

세 결과를 **같은 잣대**로 재려면 셋을 모두 노드 속성으로 저장한 뒤, 그 속성을 실은 투영을 하나 더 만들어 `gds.modularity.stats` 에 넘깁니다.

In [ ]:
# 세 알고리즘을 각각 돌려 서로 다른 속성 이름으로 저장한다
for algo, prop in [('leiden', 'community_leiden'),
                   ('louvain', 'community_louvain'),
                   ('labelPropagation', 'community_lpa')]:
    # 프로시저 이름이 값으로 바뀌므로 f-string 으로 쿼리를 만든다
    # (파라미터로는 값만 넘길 수 있고 프로시저 이름·속성 이름은 넘길 수 없다)
    out = run_cypher(f'''
        CALL gds.{algo}.write('air', {{ writeProperty: '{prop}' }})
        YIELD communityCount
        RETURN communityCount''')
    print(f'{algo:18s} 묶음 {out[0]["communityCount"]}개')

In [ ]:
# 세 결과를 같은 잣대로 재려고 세 속성을 함께 실은 투영을 하나 더 만든다
run_cypher("CALL gds.graph.drop('scored', false) YIELD graphName")
run_cypher('''
    CALL gds.graph.project('scored', 'Airport',
      { ROUTE: { orientation: 'UNDIRECTED' } },
      { nodeProperties: ['community_leiden', 'community_louvain', 'community_lpa'] })
    YIELD nodeCount RETURN nodeCount''')

compare = []
for name, prop in [('Leiden', 'community_leiden'),
                   ('Louvain', 'community_louvain'),
                   ('라벨 전파', 'community_lpa')]:
    # 모듈러리티: 그 분할이 얼마나 무리다운가. 정답이 없어도 잴 수 있는 점수다
    score = run_cypher(f'''
        CALL gds.modularity.stats('scored', {{ communityProperty: '{prop}' }})
        YIELD modularity, communityCount
        RETURN round(modularity, 4) AS modularity, communityCount''')[0]
    labels = run_cypher(f'MATCH (a:Airport) RETURN a.country AS country, a.{prop} AS community')
    found = pd.DataFrame(labels)
    compare.append({
        'algorithm': name,
        'communities': score['communityCount'],
        # 가장 큰 묶음이 전체를 삼켰는지 보려면 최대 크기도 함께 봐야 한다
        'biggest': int(found['community'].value_counts().max()),
        'modularity': score['modularity'],
        'nmi': round(normalized_mutual_info_score(found['country'], found['community']), 4),
    })
display(pd.DataFrame(compare))

> 표를 읽어 봅시다.

- **Leiden 과 Louvain** 은 묶음 수도(각각 5~8개, 6~9개), NMI 도(0.675~0.767, 0.699~0.774) 비슷합니다. **이 데이터에서는 성적이 거의 같습니다.** 그래도 Leiden 을 기준으로 두는 이유가 따로 있는데, 그건 아래에서 직접 재 봅니다.
- **라벨 전파**는 다른 자리에 있습니다. 묶음이 2개에서 6개까지 들쭉날쭉하고, NMI 는 0.104에서 0.609 로 Leiden·Louvain 보다 늘 낮습니다. 무엇보다 **가장 큰 묶음 하나가 전체의 37%에서 96%** 를 차지합니다. **여러분 화면에 묶음이 몇 개로 나오든 놀라지 마세요.** 그 이유가 바로 아래에 있습니다.

라벨 전파는 이웃의 다수결로 라벨을 퍼뜨립니다. 그래서 **큰 묶음이 더 커지기 쉽습니다.** 우리가 데이터를 42번 새로 적재하며 모두 168번 돌려 보니 묶음 수가 2개에서 6개까지 갈렸습니다. 어떤 적재에서는 가장 큰 묶음 하나가 전체의 96% 를 가져갔습니다.

**같은 적재 안에서는 몇 번을 돌려도 같은 답이 나옵니다.** 갈리는 것은 적재를 새로 했을 때입니다. 즉 이 알고리즘의 답을 정하는 것은 무작위 시작점이 아니라 **노드가 그래프에 들어간 순서**입니다. 그래서 옆자리 화면과 결과가 다를 수 있고, 같은 노트북을 처음부터 다시 실행하면 또 달라질 수 있습니다. 이 성질은 5절에서 다시 다룹니다.

여기서 배울 것은 알고리즘 순위가 아닙니다. **결과를 그대로 믿지 말고 최대 크기와 모듈러리티를 먼저 보라**는 것입니다. 표를 안 만들었다면 묶음 수만 보고 "라벨 전파로도 묶음을 잘 찾았다"고 보고했을 것입니다. 큰 묶음 하나가 절반 넘게 가져간 실행이라면 **무리를 나눴다고 하기 어려운데도** 말입니다.

> **이 표의 Leiden·Louvain 두 줄은 숫자를 그대로 옮겨 적지 마십시오.** 씨앗을 주지 않고 돌렸기 때문에 `biggest` 와 `nmi` 는 다시 실행하면 달라집니다(라벨 전파 줄은 같은 적재 안에서는 같습니다). 얼마나 달라지는지, 고정하려면 무엇을 줘야 하는지는 **5절**에서 잽니다. 지금 표에서 읽어야 할 것은 값이 아니라 **라벨 전파만 다른 자리에 있다**는 것입니다.

<img src="images/교안/세_알고리즘_비교.png" width="860">

세 알고리즘이 무리를 만드는 방식을 한 장으로 정리한 그림입니다. Leiden 과 Louvain 은 무리를 잘게 나눴다가 합치며 모듈러리티를 올리고, 라벨 전파는 이웃의 다수결로 라벨을 퍼뜨립니다. 그래서 라벨 전파는 빠른 대신 큰 묶음으로 쏠립니다.

### Leiden 을 기준으로 두는 이유 확인하기

2절에서 "Leiden 은 **속이 끊긴 묶음**이 생기지 않도록 보장한다"고 했습니다. 같은 묶음으로 묶였는데 그 안에서는 서로 이어지지 않는 경우를 말합니다. 그런 묶음은 "함께 이어진 무리"라고 부를 수 없으니 결과를 통째로 못 쓰게 만듭니다.

말로 듣고 넘어가지 말고 **직접 세어 봅시다.** 묶음 안 공항만 남긴 부분 그래프가 두 조각 이상이면 그 묶음은 속이 끊긴 것입니다.

In [ ]:
import networkx as nx

# DISTINCT 로 받아야 왕복 노선이 두 번 들어오지 않는다
link_rows = run_cypher('MATCH (a:Airport)-[:ROUTE]-(b:Airport) '
                       'RETURN DISTINCT a.iata AS a, b.iata AS b')
whole = nx.Graph((r['a'], r['b']) for r in link_rows)

for prop in ['community_leiden', 'community_louvain']:
    members = {}
    for r in run_cypher(f'MATCH (a:Airport) RETURN a.iata AS who, a.{prop} AS c'):
        members.setdefault(r['c'], []).append(r['who'])
    # 공항 한 곳짜리 묶음은 볼 것이 없으니 2곳 이상만 본다
    broken = sum(1 for ids in members.values()
                 if len(ids) > 1 and nx.number_connected_components(
                     whole.subgraph([i for i in ids if i in whole])) > 1)
    print(f'{prop:18s} 속이 끊긴 묶음 {broken}개')

> **Leiden 은 0 입니다.** 보장이 있으니 당연합니다. Louvain 도 이 데이터에서는 대개 0 이 나옵니다. 그렇다고 둘이 같은 것은 아닙니다. **Louvain 은 안 생긴다는 보장이 없고 Leiden 은 보장합니다.** 재서 0 인 것과 잴 필요가 없는 것은 다릅니다.

그래서 앞의 표를 이렇게 읽어야 합니다. **이 데이터에서는 성적이 같으니 Louvain 을 써도 되지만, 기본값으로 무엇을 둘지는 성적이 아니라 보장으로 정한다.** 여기에 5절에서 쓸 `gamma`(해상도) 설정이 Leiden 에만 있다는 점까지 더하면 기준 알고리즘은 Leiden 이 됩니다.

## 4-2. 관계에 붙은 숫자를 반영하기: `relationshipWeightProperty`

여기까지는 "노선이 있다"만 봤습니다. 관계에 **얼마나 굵은 노선인지** 같은 숫자가 붙어 있으면 그 숫자까지 커뮤니티 탐지에 넘길 수 있습니다. 우리 데이터에는 `airlines`(그 방향을 운항하는 항공사 수)가 있고, 1절에서 투영에 실어 두었습니다.

```cypher
CALL gds.leiden.stats('투영이름', { relationshipWeightProperty: '관계 속성 이름' })
YIELD modularity, communityCount
```

여기 새로 나온 것이 둘입니다.

- **`stats`**: 결과를 돌려주지도(`stream`) 저장하지도(`write`) 않고 **요약만** 냅니다. 모듈러리티 하나만 필요하면 이 한 번으로 끝납니다. 방금 세 알고리즘을 속성으로 저장하고 투영을 하나 더 만든 것은 **서로 다른 알고리즘**을 같은 잣대로 재야 했기 때문이고, **한 알고리즘의 설정을 바꿔 가며 비교할 때는 `stats` 가 훨씬 짧습니다.**
- **`relationshipWeightProperty`**: 그 속성값을 관계의 **무게**로 씁니다. 커뮤니티 탐지에서는 **값이 클수록 가까운 사이**입니다. 항공사 열 곳이 다니는 노선을 한 곳만 다니는 노선보다 촘촘하다고 보는 것입니다.

이것으로 GDS 실행 모드 **네 가지**를 모두 짚었습니다(`mutate` 는 33일차에서 직접 써 봤습니다).

| 모드 | 결과를 어디에 두나 | 언제 쓰나 |
|---|---|---|
| `stream` | 아무 데도. 그 자리에서 표로 받는다 | 눈으로 보고 파이썬으로 이어 쓸 때 |
| `stats` | 아무 데도. 요약 숫자만 | 모듈러리티 같은 점수 하나만 필요할 때 |
| `mutate` | **투영 안에** 노드 속성으로 | 그 결과를 **다음 알고리즘에 바로 먹일 때** |
| `write` | 데이터베이스에 노드 속성으로 | 나중에 평범한 Cypher 로 조회할 때 |

> 4-1 에서는 세 결과를 `write` 로 DB 에 저장한 뒤 그 속성을 실은 투영을 **하나 더** 만들었습니다. 사실 `gds.leiden.mutate('air', { mutateProperty: 'community_leiden' })` 를 쓰면 결과가 **투영 안에** 남아, DB 를 거치지도 재투영하지도 않고 `gds.modularity.stats('air', { communityProperty: 'community_leiden' })` 로 바로 잽니다. 4-1 을 길게 쓴 것은 **세 알고리즘을 나란히 두고 비교하는 과정**을 보이기 위해서였습니다. "결과를 다시 알고리즘에 먹이려면 DB 를 거쳐야 한다"고 기억하면 틀립니다.

> **인자 이름이 같다고 뜻이 같지 않습니다.** 다음 교시(교안_02)에서 볼 `gds.shortestPath.dijkstra` 는 같은 `relationshipWeightProperty` 를 **거리**로 읽습니다. 거기서는 값이 클수록 **먼** 사이입니다. 커뮤니티 탐지에 거리 값(`km`)을 그대로 넘기면 "멀수록 가깝다"가 되어 **뒤집힌 채** 무리가 나옵니다. 에러는 나지 않습니다. 넘기기 전에 그 숫자가 **무게인지 거리인지** 확인하세요.

아래에서 `airlines` 를 **실제로 넘겨 봅니다.** 씨앗과 `concurrency` 를 고정해 가중치 말고는 달라지는 것이 없게 만듭니다(이 설정이 무엇인지는 5절에서 다룹니다).

In [ ]:
# 가중치 없이 한 번, 노선 굵기(airlines)를 실어 한 번. 둘만 견주려고 씨앗을 고정한다
plain = run_cypher('''
    CALL gds.leiden.stats('air', { randomSeed: 42, concurrency: 1 })
    YIELD modularity, communityCount
    RETURN round(modularity, 4) AS modularity, communityCount''')[0]

# 같은 호출에 relationshipWeightProperty 만 더한다. 1절에서 투영에 실어 둔 속성이라야 한다
weighted = run_cypher('''
    CALL gds.leiden.stats('air', { relationshipWeightProperty: 'airlines',
                                   randomSeed: 42, concurrency: 1 })
    YIELD modularity, communityCount
    RETURN round(modularity, 4) AS modularity, communityCount''')[0]

print('가중치 없이  :', plain)
print('airlines 실어:', weighted)

> 가중치가 없을 때 모듈러리티는 **0.5545에서 0.556** 사이입니다. `airlines` 를 실으면 **0.5269** 대로 **내려갑니다.** 묶음 수는 **5개**로 나옵니다.

**가중치를 주면 점수가 오른다고 외우면 안 됩니다.** 무게를 넣으면 모듈러리티의 계산 자체가 달라집니다. 관계 개수를 세는 대신 무게를 더하기 때문입니다. 그래서 **값이 어느 쪽으로 움직이는지는 데이터가 정합니다.** 이 노선망에서는 내려갔습니다. 왜 그런지는 굵은 노선이 어디에 놓여 있는지 들여다봐야 알 수 있습니다. 중요한 것은 **재 보고 말한다**는 태도입니다.

두 분할이 얼마나 다른지도 재 봤습니다. 가중치를 준 분할과 주지 않은 분할의 NMI 는 **0.793에서 0.865** 사이였습니다(우리가 여러 번 재 본 값입니다). 완전히 다른 답도 아니고 같은 답도 아닙니다. **가중치는 설정 하나가 아니라 "무엇을 가깝다고 볼 것인가"라는 정의**입니다.

> 투영에 실려 있지 않은 속성 이름을 주면 `Relationship weight property ... not found` 에러가 납니다. 이때 고칠 것은 알고리즘이 아니라 **투영**입니다. 1절에서 방향 투영을 Leiden 에 넘겼을 때와 같은 종류의 신호입니다.

### 🖐️ 함께 따라하기: 다섯 부서 부분망에서 라벨 전파 확인하기

다섯 부서 부분망의 관계에는 숫자가 붙어 있지 않아 `relationshipWeightProperty` 를 줄 수 없습니다. 대신 **라벨 전파**를 돌려 봅니다. `'team'` 투영에 `gds.labelPropagation.stream` 을 돌리고, **묶음 수**와 **가장 큰 묶음의 인원**을 출력하세요. 부서 정답과의 NMI 도 구하고, 마지막으로 `pd.crosstab` 으로 **묶음 대 부서 교차표**를 만들어 출력하세요.

**확인 기준**: 이 그래프에서는 적재를 새로 해도 라벨 전파가 같은 답을 냈습니다(항공 노선망과 다른 점입니다). 묶음 **4개**(Leiden 의 5개보다 적습니다)·가장 큰 묶음 **96명**·NMI **0.879** 입니다. 수를 맞히는 것으로 끝내지 말고, **부서 두 개를 한 묶음으로 합쳐 버렸다**는 사실을 교차표에서 직접 확인하십시오. 한 줄에 두 부서의 인원이 나란히 들어 있는 행이 그것입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) gds.labelPropagation.stream('team') 을 YIELD nodeId, communityId 로 받는다
#    (결과는 team_lpa 라는 이름의 DataFrame 으로 둔다. NMI 도 같은 셀에서 이어서 구한다)
# 2) employee_id·dept·communityId 를 돌려받아 DataFrame 으로 만든다
# 3) 묶음 수, 가장 큰 묶음 인원(value_counts 의 최댓값), NMI 를 출력한다
# 4) pd.crosstab 으로 묶음 대 부서 교차표를 만들어 display 한다

### ✅ 바로 확인 퀴즈 (4-1·4-2)

**1.** 모듈러리티가 **0 에 가깝다**는 것은 무슨 뜻인가요?

<details><summary>정답 보기</summary>

묶음 안쪽 관계가 **무작위로 이었을 때와 다르지 않다**는 뜻입니다. 전원을 한 덩어리로 묶으면 나눈 게 없으므로 0 이 됩니다. 무리를 찾지 못했다는 신호입니다.

</details>

**2.** 세 알고리즘 결과를 비교할 때 **묶음 수만** 보면 무엇을 놓치나요?

<details><summary>정답 보기</summary>

**크기 쏠림**입니다. 라벨 전파는 묶음을 2개에서 6개 만드는데, 그중 하나가 전체의 37%에서 96% 를 가져갑니다. 적재를 새로 하면 묶음 수가 2개에서 6개까지 갈립니다. 개수와 함께 **최대 크기**와 **모듈러리티**를 봐야 합니다.

</details>

**3.** 노선 거리 `km` 를 커뮤니티 탐지와 최단 경로에 똑같이 `relationshipWeightProperty` 로 넘겼습니다. 어느 쪽이 뜻이 뒤집힌 채 돌아갈까요? 에러가 나나요?

<details><summary>정답 보기</summary>

**커뮤니티 탐지 쪽**입니다. 커뮤니티 탐지는 값이 클수록 가까운 사이로 읽으므로 "멀수록 가깝다"가 됩니다. 최단 경로는 같은 값을 거리로 읽으니 제대로 돕니다. **에러는 나지 않습니다.** 숫자이기만 하면 그대로 돌아가므로, 넘기기 전에 그 값이 **무게인지 거리인지** 사람이 확인해야 합니다.

</details>

---
# 5. 결과가 실행마다 흔들린다

이걸 모르고 넘어가면 곤란한 일이 생깁니다. 어제 만든 보고서의 "권역 여덟 개"가 오늘 다시 돌리면 일곱 개가 되고, 어느 쪽이 맞는지 설명할 수 없게 됩니다. **얼마나 흔들리는지 먼저 재고**, 필요하면 **고정하는 방법**을 알아야 합니다.

## 5-1. 얼마나 흔들리는지 재 보기

### 왜 필요할까요?
여기까지 오면서 **묶음마다 공항 수가** 조금씩 달라졌을 수 있습니다. 오류가 아닙니다. **커뮤니티 탐지 알고리즘은 시작점을 무작위로 고르고, 여러 갈래로 나눠 동시에 계산합니다.** 그래서 같은 데이터·같은 설정이어도 답이 조금씩 다릅니다.

In [ ]:
# 같은 설정으로 다섯 번 돌려 본다. 답이 얼마나 흔들리는지 눈으로 확인하는 것이 목적
runs = []
for _ in range(5):
    # 씨앗을 주지 않았으니 매번 다른 시작점에서 출발한다
    rows = run_cypher('''
        CALL gds.leiden.stream('air')
        YIELD nodeId, communityId
        RETURN gds.util.asNode(nodeId).country AS country, communityId AS community''')
    one = pd.DataFrame(rows)
    runs.append({'communities': one['community'].nunique(),
                 'biggest': int(one['community'].value_counts().max()),
                 'nmi': round(normalized_mutual_info_score(one['country'],
                                                           one['community']), 4)})
display(pd.DataFrame(runs))

> 세 열이 서로 다른 이야기를 합니다. 여기가 이 절의 핵심입니다.

- `communities` 는 **거의 안 흔들립니다.** 우리가 여러 번 돌려 보니 **5개에서 8개** 사이를 벗어나지 않았고, 가장 자주 나온 것은 7개였습니다.
- `biggest` 는 **크게 흔들립니다.** 같은 실행들에서 가장 큰 묶음의 공항 수가 **170곳에서 211곳**까지, 즉 40곳 넘게 오갔습니다.
- `nmi` 도 **0.675 에서 0.767** 사이에서 움직였습니다.

**개수가 비슷하다고 같은 답이 아닙니다.** 묶음 수는 한두 개만 움직이지만, 그 7개 안팎에 **어느 공항이 어느 묶음으로 들어가는지**가 매번 달라집니다. 가장 큰 묶음이 170곳인 날과 211곳인 날은 권역 지도가 다르게 그려집니다. 흔들리는 것은 **개수가 아니라 구성**입니다.

> 위 숫자는 **우리가 이 데이터·이 버전에서 재 본 값**입니다. 여러분 화면에는 조금 다른 값이 나올 수 있습니다. 외울 것은 값이 아니라 **직접 여러 번 돌려 폭을 재고, 그 폭을 함께 보고한다**는 절차입니다.

이걸 알고 나면 보고하는 방식이 달라집니다. "묶음 7개"는 틀린 말이 아니지만 **그것만 쓰면 안 됩니다.** 여러 번 돌려 본 뒤 **가장 큰 묶음의 폭까지 함께** 적어야 읽는 사람이 이 결과를 얼마나 믿어도 되는지 압니다.

## 5-2. 씨앗과 동시 실행 수로 고정하기

### 고정하는 법 1: `randomSeed` 만으로는 부족하다
많은 알고리즘이 무작위 시작점을 쓰기 때문에 **씨앗**(`randomSeed`)을 주면 같은 답이 나올 것 같습니다. 정말 그런지 확인해 봅시다.

In [ ]:
# 씨앗만 주고 세 번 돌려 본다. 이것만으로 고정되는지 확인하는 것이 목적
seeded = []
for _ in range(3):
    rows = run_cypher('''
        CALL gds.leiden.stream('air', { randomSeed: 42 })
        YIELD nodeId, communityId
        RETURN gds.util.asNode(nodeId).country AS country, communityId AS community''')
    one = pd.DataFrame(rows)
    seeded.append({'communities': one['community'].nunique(),
                   'biggest': int(one['community'].value_counts().max()),
                   'nmi': round(normalized_mutual_info_score(one['country'],
                                                             one['community']), 4)})
display(pd.DataFrame(seeded))

> 씨앗을 줬는데도 **세 줄이 똑같지 않을 수 있습니다.** 특히 `biggest` 를 보세요. 5-1 에서 말한 대로 **흔들리는 것은 개수가 아니라 구성**이고, 씨앗은 그 구성을 혼자서는 고정하지 못합니다.

이유는 **병렬 실행**입니다. GDS 는 그래프를 여러 조각으로 나눠 여러 스레드가 동시에 계산하는데, 어느 스레드가 먼저 끝나느냐가 매번 달라집니다. 씨앗은 "무작위 수열"만 고정할 뿐 **스레드가 합쳐지는 순서**는 고정하지 못합니다.

### 고정하는 법 2: `concurrency: 1` 을 함께 준다
스레드를 하나로 줄이면 합쳐지는 순서가 정해집니다. 씨앗과 함께 주면 그제서야 재현됩니다.

In [ ]:
# randomSeed 와 concurrency 를 함께 고정하고 세 번 돌려 본다
fixed = []
for _ in range(3):
    # concurrency: 1 은 스레드를 하나로 줄여 결과가 합쳐지는 순서까지 정한다(대신 느리다)
    rows = run_cypher('''
        CALL gds.leiden.stream('air', { randomSeed: 42, concurrency: 1 })
        YIELD nodeId, communityId
        RETURN gds.util.asNode(nodeId).country AS country, communityId AS community''')
    one = pd.DataFrame(rows)
    # 앞 셀과 같은 세 열이라야 무엇이 달라졌는지 그 자리에서 견줄 수 있다
    fixed.append({'communities': one['community'].nunique(),
                  'biggest': int(one['community'].value_counts().max()),
                  'nmi': round(normalized_mutual_info_score(one['country'],
                                                            one['community']), 4)})
display(pd.DataFrame(fixed))

> 이번에는 `biggest` 까지 **세 줄이 같습니다.** 앞 셀과 열 구성이 같으니 무엇이 달라졌는지 그 자리에서 견주면 됩니다.

**여기서 한 가지를 더 알아야 합니다. 씨앗은 '이 투영 위에서' 만 보장합니다.** 우리가 데이터를 지우고 처음부터 다시 적재해 다시 투영한 뒤 같은 씨앗으로 재 봤더니, 묶음이 **7개에서 8개** 사이로 갈렸습니다(NMI 0.707에서 0.767). 노드가 그래프에 들어간 **순서**가 달라지면 내부 번호도 달라지고, 알고리즘이 훑는 차례도 달라지기 때문입니다.

> **4-1 의 라벨 전파가 이 성질의 극단적인 예입니다.** 같은 적재 안에서는 몇 번을 돌려도 답이 같은데, 적재를 새로 하면 묶음이 2개에서 6개까지 갈립니다. 흔들림의 원인이 무작위가 아니라 **노드가 들어간 순서**라는 뜻입니다. 라벨 전파에는 고정할 씨앗 자체가 없습니다(`randomSeed` 를 주면 `Unexpected configuration key` 에러가 납니다). **적재 절차까지 함께 적어 두어야** 남이 같은 답을 얻습니다.

**실무 지침 네 가지.**

1. **재현이 필요하면 `randomSeed` 와 `concurrency: 1` 을 함께** 줍니다. 하나만으로는 안 됩니다.
2. 그렇게 해도 **적재를 새로 하면 또 갈립니다.** 재현은 "같은 데이터·같은 적재·같은 투영" 까지 묶어야 성립합니다.
3. `concurrency: 1` 은 **느립니다.** 재현이 필요한 최종 산출물에만 쓰고, 탐색 단계에서는 그냥 돌립니다.
4. **고정한 답이 더 좋은 답은 아닙니다.** 씨앗을 고정해 얻은 답도 여러 후보 중 하나일 뿐입니다. 중요한 결론은 **여러 번 돌려서 반복되는 것**만 씁니다.

> 참고: 이 버전의 **Louvain 에는 `randomSeed` 가 아예 없습니다.** 넣으면 `Unexpected configuration key: randomSeed` 에러가 납니다(`gamma` 도 마찬가지로 없습니다). 그래도 Louvain 은 `concurrency: 1` 만으로 재현됩니다. 정말 그런지 아래에서 확인합니다.

In [ ]:
# Louvain 은 randomSeed 를 받지 않는다. concurrency 만으로 재현되는지 직접 확인한다
# Leiden 은 씨앗까지 필요했다. 재현에 필요한 설정이 알고리즘마다 다르다
louvain_fixed = []
for _ in range(3):
    rows = run_cypher('''
        CALL gds.louvain.stream('air', { concurrency: 1 })
        YIELD nodeId, communityId
        RETURN gds.util.asNode(nodeId).country AS country, communityId AS community''')
    one = pd.DataFrame(rows)
    louvain_fixed.append({'communities': one['community'].nunique(),
                          'biggest': int(one['community'].value_counts().max()),
                          'nmi': round(normalized_mutual_info_score(one['country'],
                                                                    one['community']), 4)})
display(pd.DataFrame(louvain_fixed))

> 여기서도 세 열이 모두 같습니다. **재현에 필요한 설정은 알고리즘마다 다릅니다.** 어떤 것은 씨앗이 있고 어떤 것은 없습니다. "씨앗을 줬으니 됐다"고 넘기지 말고, 이렇게 **몇 번 돌려서 같은지 직접 확인**하는 습관이 재현 가능한 분석을 만듭니다.

In [ ]:
# 라벨 전파도 세 번 돌려 본다. 씨앗도 concurrency 도 주지 않는다
# 4-1 에서 '같은 적재 안에서는 늘 같은 답' 이라고 했다. 말이 아니라 표로 확인한다
lpa_repeat = []
for _ in range(3):
    rows = run_cypher('''
        CALL gds.labelPropagation.stream('air')
        YIELD nodeId, communityId
        RETURN gds.util.asNode(nodeId).country AS country, communityId AS community''')
    one = pd.DataFrame(rows)
    lpa_repeat.append({'communities': one['community'].nunique(),
                       'biggest': int(one['community'].value_counts().max()),
                       'nmi': round(normalized_mutual_info_score(one['country'],
                                                                 one['community']), 4)})
display(pd.DataFrame(lpa_repeat))

> 라벨 전파도 **같은 적재 안에서는 세 줄이 같습니다.** 아무 설정도 주지 않았는데도 그렇습니다. 이 알고리즘의 답을 정하는 것은 난수가 아니라 노드를 훑는 순서이고, 그 순서는 적재할 때 정해집니다. 그래서 4-1 의 폭(묶음 2개에서 6개)은 **새로 적재할 때** 생깁니다. 같은 표가 나왔다고 재현된다고 믿으면, 다음 주에 데이터를 다시 적재한 동료가 다른 답을 들고 옵니다.

## 5-3. 묶음 수를 정하는 값: 해상도(gamma)

3절에서 "국가·지역은 47개인데 묶음은 몇 개뿐"이라는 문제가 남아 있었습니다. **더 잘게 쪼개라고 지시할 수 있습니다.** Leiden 의 `gamma`(해상도)가 그 값입니다.

- `gamma` 를 **올리면** 묶음 안쪽에 요구하는 촘촘함이 높아져 **더 잘게** 갈립니다.
- `gamma` 를 **내리면** 느슨해져 **더 크게 뭉칩니다.**
- 기본값은 1.0 입니다.

얼마가 맞는 값일까요. **정답은 데이터마다 다릅니다.** 그래서 스윕(여러 값을 훑어보기)을 합니다. 여기서는 정답 국가·지역이 있으니 NMI 도 함께 재서, 어느 값이 지도에 가장 가까운지 봅니다.

In [ ]:
# gamma 를 바꿔 가며 묶음 수와 정답 일치도를 함께 본다.
# 값을 비교하는 게 목적이므로 흔들림을 없애려고 씨앗과 concurrency 를 고정한다
sweep = []
# 기본값 1.0 을 가운데 두고 아래위로 훑는다. 올리면 잘게 갈리고 내리면 크게 뭉친다
for gamma in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]:
    rows = run_cypher('''
        CALL gds.leiden.stream('air', { gamma: $gamma, randomSeed: 42, concurrency: 1 })
        YIELD nodeId, communityId
        RETURN gds.util.asNode(nodeId).country AS country, communityId AS community''',
                      gamma=gamma)
    one = pd.DataFrame(rows)
    counts = one['community'].value_counts()
    sweep.append({'gamma': gamma,
                  'communities': len(counts),
                  'biggest': int(counts.max()),
                  'nmi': round(normalized_mutual_info_score(one['country'],
                                                            one['community']), 4)})
display(pd.DataFrame(sweep))

> 읽어 봅시다. 아래 폭은 우리가 여러 번 재 본 값입니다.

- `gamma` 를 0.5 로 내리면 묶음이 **4개에서 6개**로 줄고 NMI 도 **0.583** 대로 뚝 떨어집니다. **너무 뭉쳤습니다.** 권역 여럿이 한 덩어리가 된 것입니다.
- 기본값 1.0 에서는 묶음 **7개에서 8개**, NMI **0.707에서 0.767** 입니다.
- 3.0 까지 올리면 묶음이 **28개에서 31개**로 잘게 부서집니다. NMI 는 0.792에서 0.808 입니다.
- 이 스윕에서 NMI 가 가장 높았던 것은 gamma **2.5 또는 3.0**(묶음 21~31개, NMI 최고 0.809)입니다. **어느 쪽이 최고인지는 적재에 따라 갈립니다.**

**gamma 를 올리면 권역이 국가로 쪼개집니다.** 기본값에서는 여러 나라가 한 권역으로 묶여 있다가, 값을 올릴수록 나라 단위로 갈라집니다. 국가·지역과의 NMI 가 함께 오르는 것이 그 증거입니다.

> 묶음 수가 정답 라벨 수(47개)에 가장 가까워지는 것은 gamma **3.0**(묶음 28~31개)이고, NMI 가 가장 높은 것은 gamma **2.5 또는 3.0** 입니다. 둘은 겹칠 때도 있고 아닐 때도 있습니다. **개수가 맞는다고 가장 잘 맞는 분할이라는 보장은 없습니다.** 개수는 고르는 기준이 되지 못합니다.

> 숫자는 **장비와 적재 순서**에 따라 조금 다릅니다. 이 노트북 안에서 다시 돌리면 씨앗과 `concurrency: 1` 을 줬으니 같은 값이 나옵니다. 외울 것은 값이 아니라 **방향**(gamma 를 올리면 잘게 갈린다)입니다.

**NMI 가 가장 높은 값을 그냥 고르면 될까요?** 아닙니다. 잘게 쪼갤수록 NMI 는 대체로 오르지만, 쓸모는 함께 오르지 않습니다. 묶음이 28개를 넘으면 "권역을 찾았다"는 말이 무색해집니다. **무엇에 쓸 것인지**가 값을 정합니다. 권역 단위로 보려면 기본값 언저리가, 나라 단위로 보려면 높은 값이 맞습니다.

### 🖐️ 함께 따라하기: 다섯 부서 부분망의 해상도 바꿔 보기

`'team'` 투영에서 `gamma` 를 **0.5 와 3.0** 두 값으로 돌려, 각각 **묶음 수**와 **부서 정답과의 NMI** 를 출력하세요. 흔들림을 없애려면 `randomSeed: 42, concurrency: 1` 을 함께 주세요.

**확인 기준**: `gamma=0.5` 은 묶음 **4개**·NMI **0.879**, `gamma=3.0` 은 묶음 **14개**·NMI **0.767** 입니다. 기본값 1.0 일 때의 0.963 보다 **양쪽 다 나쁩니다.** 이 그래프에서는 기본값이 이미 좋은 값이었다는 뜻입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) [0.5, 3.0] 을 for 문으로 돌면서 gds.leiden.stream('team', {gamma: $g,
#    randomSeed: 42, concurrency: 1}) 을 실행한다 (gamma 는 파라미터로 넘긴다)
# 2) dept 와 communityId 를 받아 DataFrame 으로 만든다
# 3) 묶음 수와 NMI 를 한 줄씩 출력한다

### ✅ 바로 확인 퀴즈

**1.** "`randomSeed` 를 줬으니 결과는 항상 같다"는 말이 왜 틀렸나요?

<details><summary>정답 보기</summary>

GDS 는 여러 스레드로 동시에 계산하는데, 씨앗은 무작위 수열만 고정할 뿐 **스레드가 합쳐지는 순서**는 고정하지 못합니다. `concurrency: 1` 을 함께 줘야 합니다. 그렇게 해도 **데이터를 다시 적재하면 또 갈립니다.**

</details>

**2.** 묶음이 너무 크게 뭉쳐 쓸모가 없습니다. 어느 값을 어느 방향으로 바꿔야 할까요?

<details><summary>정답 보기</summary>

`gamma` 를 **올립니다.** 해상도가 높아져 더 잘게 갈립니다.

</details>

**3.** NMI 가 가장 높은 `gamma` 를 고르면 항상 최선인가요?

<details><summary>정답 보기</summary>

아닙니다. 잘게 쪼갤수록 NMI 는 대체로 오르지만 묶음이 수십 개가 되면 해석과 활용이 어려워집니다. **무엇에 쓸 결과인지**를 함께 봐야 합니다. 정답 라벨 개수에 맞추는 것도 기준이 되지 못합니다.

</details>

---
# 6. 공항 767곳짜리 그래프를 읽을 수 있는 그림으로

## 6-1. 먼저 줄이고 나서 그리기

### 왜 필요할까요?
노드가 열댓 개면 점과 선을 그대로 그려도 읽힙니다. **공항 767곳에 노선 관계가 8,045개면 그대로 그린 그림은 새까만 털뭉치가 됩니다.** 그래도 그림은 필요합니다. 표로는 안 보이는 쏠림과 이웃 관계가 그림에서는 한눈에 보이기 때문입니다.

방법은 **줄이는 것**입니다. 오늘은 세 가지를 씁니다.

1. **크기 막대**: 묶음마다 공항 수만 세어 막대로 그린다. 쏠림이 바로 보인다.
2. **위경도 산점도**: 공항의 좌표를 그대로 찍고 묶음으로 색을 나눈다. 선을 하나도 그리지 않아 털뭉치가 되지 않는다.
3. **메타그래프**: 묶음 하나를 점 하나로 줄이고, 묶음 사이 노선 수를 선 굵기로 그린다(6-2).

> 앞의 두 그림은 2-1 의 `stream` 결과(`community` 표)를, 메타그래프는 2-2 에서 `write` 로 저장한 결과를 씁니다. **서로 다른 실행**이라 5절에서 본 대로 번호도 구성도 조금 다릅니다. 두 그림을 겹쳐 읽지 말고 각각 무엇을 보여 주는지로 읽으세요.

In [ ]:
import platform

import matplotlib.pyplot as plt

# 한글 폰트: 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == "Windows":
    KOREAN_FONT = "Malgun Gothic"
elif platform.system() == "Darwin":          # macOS
    KOREAN_FONT = "AppleGothic"
else:                                        # Linux (Colab 등)
    KOREAN_FONT = "NanumGothic"

plt.rcParams["font.family"] = KOREAN_FONT    # 이후 모든 그림에 이 폰트가 적용된다
plt.rcParams["axes.unicode_minus"] = False   # 마이너스(-) 부호 깨짐 방지

# 1) 묶음 크기 막대. 큰 것부터 세우면 쏠림이 바로 보인다
ordered = sizes.sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(9, 4))
# x 는 크기 순위(0부터), y 는 공항 수. 묶음 번호는 뜻이 없으니 축에 쓰지 않는다
ax.bar(range(len(ordered)), ordered.values, color='#4C72B0')
ax.set_xlabel('묶음 (큰 순서)', fontfamily=KOREAN_FONT)
ax.set_ylabel('공항 수', fontfamily=KOREAN_FONT)
ax.set_title('묶음 크기 분포', fontfamily=KOREAN_FONT)
plt.show()

<img src="images/교안/아시아_공항_지도.png" width="820">

아래 셀이 그리는 그림입니다. **지도를 그린 것이 아닙니다.** 공항의 경도를 x, 위도를 y 로 찍은 산점도일 뿐인데 대륙 모양이 나옵니다. 색은 알고리즘이 나눈 묶음입니다.

In [ ]:
# 2) 위경도 산점도. x 는 경도, y 는 위도, 색은 묶음이다
fig, ax = plt.subplots(figsize=(9, 6))
# 큰 묶음부터 그려야 범례 순서가 크기 순으로 선다. 묶음 번호는 이름표로 쓰지 않는다
for rank, community_id in enumerate(sizes.index, start=1):
    part = community[community['community'] == community_id]
    ax.scatter(part['lon'], part['lat'], s=14, alpha=0.85, label=f'{rank}위 {len(part)}곳')
# 위도 1도와 경도 1도를 같은 길이로 그려야 대륙 모양이 찌그러지지 않는다
ax.set_aspect('equal')
ax.set_xlabel('경도', fontfamily=KOREAN_FONT)
ax.set_ylabel('위도', fontfamily=KOREAN_FONT)
ax.set_title('공항 위치와 묶음 (점 하나가 공항 하나)', fontfamily=KOREAN_FONT)
ax.legend(prop={'family': KOREAN_FONT, 'size': 8}, ncol=2)
plt.show()

> 이 그림이 오늘의 결론을 한 장으로 보여 줍니다. **알고리즘은 좌표를 보지 않았습니다.** 노선 연결만 보고 나눴는데 **색이 지리적으로 붙어 있습니다.** 3절에서 잰 NMI 0.738 를 눈으로 보는 셈입니다.

동시에 **색 경계가 국경과 정확히 맞지는 않는다**는 것도 보입니다. 한 색이 여러 나라에 걸쳐 있고, 큰 나라 하나가 통째로 한 색인 곳도 있습니다. 묶음은 나라가 아니라 **권역**이라는 3절의 읽기가 그림에서도 그대로 확인됩니다.

> 색과 순위는 실행할 때마다 달라집니다. **모양**을 보세요.

## 6-2. 메타그래프로 묶음 사이의 흐름 보기

<img src="images/교안/커뮤니티_메타그래프.png" width="640">

**메타그래프**는 묶음 하나를 점 하나로 줄인 그림입니다. 점의 크기는 공항 수, 선의 굵기는 두 묶음 사이에 오가는 노선 수입니다. 767개의 점이 **손에 꼽을 만큼**으로 줄어 **노선망 전체의 뼈대**가 보입니다.

여기서는 2-2 에서 `write` 로 저장해 둔 `community` 속성을 씁니다. GDS 없이 평범한 Cypher 로 묶음 사이 노선을 셀 수 있다는 것이 `write` 를 쓰는 이유였습니다.

In [ ]:
import networkx as nx

# a.community <> b.community 조건이 묶음 안쪽 노선은 빼고 '사이'만 남긴다
meta_rows = run_cypher('''
    MATCH (a:Airport)-[:ROUTE]->(b:Airport)
    WHERE a.community <> b.community
    RETURN a.community AS source, b.community AS target, count(*) AS routes''')

# 노드에 저장해 둔 community 로 다시 읽는다
# 위 sizes 는 2-1 의 stream 결과라 다른 실행이다. 번호뿐 아니라 구성도 조금 다르다
sizes_now = pd.Series(
    {r['community']: r['airports'] for r in run_cypher(
        'MATCH (a:Airport) RETURN a.community AS community, count(*) AS airports')})

meta = nx.Graph()
for community_id, airports in sizes_now.items():
    meta.add_node(community_id, airports=int(airports))
for r in meta_rows:
    # 무방향으로 합치므로 이미 있는 선이면 노선 수를 더한다
    before = meta.edges[r['source'], r['target']]['routes'] if meta.has_edge(
        r['source'], r['target']) else 0
    meta.add_edge(r['source'], r['target'], routes=before + r['routes'])
print('메타그래프 노드:', meta.number_of_nodes(), '· 선:', meta.number_of_edges())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
# seed 로 배치를 고정해 다시 그려도 같은 모양이 나오게 한다
pos = nx.spring_layout(meta, seed=42, k=1.2)
airports_of = [meta.nodes[n]['airports'] for n in meta.nodes()]
routes_of = [meta.edges[e]['routes'] for e in meta.edges()]
biggest_flow = max(routes_of)
nx.draw_networkx_edges(meta, pos, ax=ax, edge_color='#AAAAAA',
                       width=[6 * m / biggest_flow for m in routes_of])
nx.draw_networkx_nodes(meta, pos, ax=ax, node_color='#4C72B0',
                       node_size=[12 * p for p in airports_of], alpha=0.85)
# 점 안에는 묶음 번호 대신 공항 수를 쓴다. 번호에는 뜻이 없기 때문이다
nx.draw_networkx_labels(meta, pos, ax=ax, font_size=9, font_color='white',
                        labels={n: str(meta.nodes[n]['airports']) for n in meta.nodes()})
ax.set_title('묶음 메타그래프 (숫자는 공항 수, 선 굵기는 오가는 노선 수)',
             fontfamily=KOREAN_FONT)
ax.axis('off')
plt.show()

> 묶음끼리는 **대부분 이어져 있습니다.** 두 셀 위에서 찍은 `메타그래프 노드` 와 `선` 두 수를 견줘 보세요. 노드가 n 개면 만들 수 있는 쌍은 n x (n - 1) / 2 개입니다. 선 수가 그 값에 얼마나 가까운지 보세요. 권역이 달라도 큰 공항끼리는 어떻게든 노선이 오가기 때문입니다.

그래서 읽을 것은 **이어졌는가가 아니라 선 굵기**입니다. 굵은 선 몇 개가 이 노선망의 큰 흐름이고, 실 같은 선은 한두 편 오가는 사이입니다. **이 그림 하나가 "이 지역은 몇 개 권역이고, 그중 어디와 어디가 굵게 오가는가"에 답합니다.** 공항 767곳을 그대로 그렸다면 이것조차 못 읽었을 것입니다.

실무에서 큰 그래프를 그릴 때 기억할 것: **먼저 줄이고 나서 그립니다.** 줄이는 방법은 집계(메타그래프), 표본(상위 몇 개만), 요약(막대·산점도) 세 가지입니다.

### 🖐️ 함께 따라하기: 다섯 부서 부분망의 묶음 크기 그리기

`team_community` 의 묶음 크기를 **막대그래프**로 그리세요. 큰 순서로 세우고, 제목·축 라벨을 한글로 답니다(`fontfamily=KOREAN_FONT` 를 잊지 마세요).

**확인 기준**: 막대가 **5개** 서고, 높이는 **51·45·38·32·28** 입니다. 항공 노선망과 달리 **크기가 고르게 나뉘어** 있습니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) team_community['community'].value_counts() 를 내림차순 정렬한다
# 2) fig, ax = plt.subplots() 로 새 그림을 만든다 (앞 그림에 겹쳐 그리지 않도록)
# 3) ax.bar 로 막대를 그리고 제목·축 라벨에 fontfamily=KOREAN_FONT 를 준다

### ✅ 바로 확인 퀴즈

**1.** 공항 767곳짜리 노선망을 `spring_layout` 으로 그대로 그리면 무엇이 문제인가요?

<details><summary>정답 보기</summary>

점과 선이 겹쳐 **아무것도 읽히지 않습니다.** 그리기 전에 집계·표본·요약으로 먼저 줄여야 합니다. 위경도 산점도가 읽혔던 것은 **선을 하나도 그리지 않았기** 때문이기도 합니다.

</details>

**2.** 메타그래프에서 두 점을 잇는 선이 굵다는 것은 무슨 뜻인가요?

<details><summary>정답 보기</summary>

두 묶음 **사이에 오가는 노선이 많다**는 뜻입니다. 서로 다른 권역인데 굵게 이어져 있다면 그 사이에 환승 거점이 있을 가능성이 큽니다.

</details>

---
## 🚀 응용 클론코딩: 묶음 한 줄 요약 만들기

지금까지 한 것을 **함수 하나**로 묶습니다. 묶음 번호를 넣으면 그 묶음의 공항 수·대표 국가·지역·허브 공항을 담은 사전을 돌려주는 `describe_community(community_id)` 를 만드세요.

돌려줄 사전의 키는 `community`, `airports`, `top_country`, `hub` 네 개입니다. `top_country` 는 공항이 가장 많은 국가·지역, `hub` 는 그 묶음에서 **차수가 가장 큰 공항**의 `iata` 입니다.

6-2 에서 만든 `sizes_now` 로 **가장 큰 묶음**과 **가장 작은 묶음**을 골라 각각 적용해 출력하세요.

In [ ]:
# 🚀 응용 (아래 순서대로 직접 작성해 보세요)
# 1) def describe_community(community_id): 로 함수를 만든다
# 2) 첫 줄에서 community_id 를 int() 로 감싼다
#    (판다스 정수를 그대로 넘기면 드라이버가 받지 못한다)
# 3) a.community = $community_id 인 공항의 iata·country·차수를 차수 내림차순으로 받는다
#    (차수는 COUNT { (a)--() } 로 센다)
# 4) community·airports·top_country·hub 네 키를 가진 사전을 돌려준다
# 5) sizes_now.idxmax()·sizes_now.idxmin() 으로 가장 큰 묶음과 가장 작은 묶음에
#    각각 적용해 출력한다

---
## 이번 강의 정리

| 한 일 | 쓴 것 | 기억할 것 |
|---|---|---|
| 무방향 투영 | `gds.graph.project` + `orientation: 'UNDIRECTED'` | 관계 수가 두 배로 나오는 게 정상. 차수는 상대 공항 수와 다르다. Leiden 은 방향 투영을 거부한다 |
| 묶음 찾기 | `gds.leiden.stream` / `.write` | 개수보다 **크기 분포**를 먼저 본다 |
| 정답 대조 | `pd.crosstab`, 순도, `normalized_mutual_info_score` | 정답이 있으면 반드시 잰다. 안 맞는 부분이 오히려 발견이다 |
| 알고리즘 비교 | `gds.modularity.stats` | 라벨 전파는 큰 묶음으로 쏠리고 적재마다 답이 갈린다. 최대 크기를 확인한다 |
| 요약만 받기 | `gds.leiden.stats` | 모듈러리티 하나만 필요하면 저장도 재투영도 없이 |
| 관계의 숫자 반영 | `relationshipWeightProperty` | 커뮤니티에서는 클수록 **가깝다**. 최단 경로에서는 반대다. 값이 어느 쪽으로 움직일지는 재 봐야 안다 |
| 재현 | `randomSeed` + `concurrency: 1` | 씨앗만으로는 고정되지 않고, 새로 적재하면 또 갈린다 |
| 묶음 수 조절 | Leiden 의 `gamma` | 올리면 잘게, 내리면 크게. 정답 개수에 맞추는 것과 잘 맞는 것은 다르다 |
| 큰 그래프 그리기 | 막대·산점도·메타그래프 | **먼저 줄이고 나서 그린다** |

**오늘의 발견.** 알고리즘이 찾은 묶음은 **나라가 아니라 권역**입니다. 국가·지역은 47개인데 묶음은 5개에서 8개고, 국가·지역과 대조한 NMI 는 0.738 안팎입니다. **권역 단위로는 잘 맞고 국가 경계와는 어긋납니다.** 어긋난 자리가 실수가 아니라, 국경보다 **노선이 실제로 어떻게 이어져 있는지**를 보여 줍니다.

> 출처: OpenFlights(openflights.org) `airports.dat`·`routes.dat`. 세계 공항과 그 사이 노선을 모은 공개 데이터로, 라이선스는 ODbL(출처 표기·동일 조건 공유)입니다. 노선 자료는 2014년 6월에 갱신이 멈춘 역사 자료라 지금 운항 여부와 다를 수 있고, 시간표가 아니라 '노선이 있었는가' 만 담겨 있습니다.
>
> 출처: SNAP(snap.stanford.edu) `email-Eu-core`. 유럽의 큰 연구기관에서 오간 사내 메일을 익명화한 공개 데이터입니다. 별도 라이선스 표기가 없어 학습용으로만 씁니다. 관련 논문: Yin, Benson, Leskovec, Gleich (KDD 2017) · Leskovec, Kleinberg, Faloutsos (TKDD 2007).

## ⏭️ 예고: 다음 교시

묶음을 찾았으니 이제 **공항 하나하나의 관계**를 봅니다.

- **노드 유사도**: 인천(ICN)과 **노선 집합이 닮은 공항**은 어디인가. 나라가 달라도 닮을 수 있습니다.
- **최단 경로**: 인천에서 어느 공항까지 **몇 편**에 닿는가.
- **세 잣대**: 같은 목적지라도 **편 수 최소·거리 최소·시간 최소**가 서로 다른 길입니다. 오늘 투영에 함께 실어 둔 `km`·`hours` 를 거기서 씁니다.